In [4]:
# input
selected_foldcomp = "./tmp/metaldb_uniprot_v4"
selected_id = "./tmp/structRepId-entryId.tsv"
pred_file = "../../predict_afdb/data/pred_ge_3_clique_3.tsv"
# output
selected_pre_org_site = "./data/seqId-site-minPlddt.tsv"

In [ ]:
from tqdm import tqdm

import foldcomp
from itertools import combinations, product
from typing import List
import networkx as nx
import pandas as pd
from Bio.PDB.PDBParser import PDBParser
from Bio.PDB.Residue import Residue
from Bio.PDB.Atom import Atom

allowed_coord_atom = {
    ("Cys", "SG"),
    ("Asp", "OD1"),
    ("Asp", "OD2"),
    ("Glu", "OE1"),
    ("Glu", "OE2"),
    ("His", "ND1"),
    ("His", "NE2"),
}

def parse_pdb_str(
    id: str,
    pdb_str: str,
    positions: List[int],
    ssbond_dist_threshold: float = 2.5
) -> list[Residue]:

    # build structure
    lines = pdb_str.split("\n")
    result = []
    for l in lines:
        if l.startswith("ATOM"):
            resseq = int(l[22:26].split()[0])
            resname = l[17:20].strip()
            posi = resseq - 1
            if posi in positions or resname == "CYS":
                result.append(l)

    parser = PDBParser(QUIET=True)
    sb = parser.structure_builder
    sb.init_structure(id)
    parser._parse(result)
    sb.set_header(parser.get_header())

    cys_residues = set()
    residues = set()
    for r in sb.get_structure().get_residues():
        r: Residue
        posi = r.id[1] - 1
        if posi in positions:
            residues.add(r)
        if str.upper(r.resname) == "CYS":
            cys_residues.add(r)

    # remove ss-bond related residues
    result = []
    for r in residues:
        if str.upper(r.resname) != "CYS": 
            result.append(r)
            continue
        
        has_ss_bond = False
        for r1 in cys_residues:
            if r == r1: continue
            if r["SG"] - r1["SG"] <= ssbond_dist_threshold:
                has_ss_bond = True
                break
        
        if not has_ss_bond:
            result.append(r)

    return result

def get_coord_atom(
    r: Residue
) -> list[Atom]:
    resname = str.capitalize(r.get_resname())
    if resname == "Cys":
        return [r['SG']]
    elif resname == "Asp":
        return [r['OD1'], r['OD2']]
    elif resname == "Glu":
        return [r['OE1'], r['OE2']]
    else:
        return [r['ND1'], r['NE2']]

def filter_by_clique(
    residues: list[Residue],
    dist_min: float = 2.5,
    dist_max: float = 7.0,
    num_clique_member: int = 3,
) -> tuple[dict, list[list[int]]]:
    g = nx.Graph()
    for r1, r2 in combinations(residues, 2):
        for a1, a2 in product(get_coord_atom(r1), get_coord_atom(r2)):
            dist = a1 - a2
            if dist_min <= dist <= dist_max:
                g.add_edge(a1, a2)

    sites = set()
    cliques = nx.algorithms.find_cliques(g)
    for c in cliques:
        if len(c) >= num_clique_member:
            site = [a.parent.id[1] - 1 for a in c]
            site.sort()
            sites.add(",".join([str(i) for i in site]))

    return sites

def plddt_stat(
    pdb_str: str,
) -> list[float]:
    lines = pdb_str.split("\n")
    result = []
    for l in lines:
        if l.startswith("ATOM"):
                
            ### ref to biopython
            fullname = l[12:16]
            # get rid of whitespace in atom names
            split_list = fullname.split()
            if len(split_list) != 1:
                # atom name has internal spaces, e.g. " N B ", so
                # we do not strip spaces
                name = fullname
            else:
                # atom name is like " CA ", so we can strip spaces
                name = split_list[0]
                
            if name == "CA":
                result.append(float(l[60:66]))
    return result

In [6]:
df = pd.read_table(pred_file)
target_ids = set(pd.read_table(selected_id, header=None)[1].map(lambda x: f"AFDB:AF-{x}-F1"))
df = df[df['seq_id'].map(lambda x: x in target_ids)]
seq_id_to_positions = dict(
    zip(
        df["seq_id"].map(lambda x: x.removeprefix("AFDB:") + "-model_v4"),
        df["posi"].map(lambda x: [int(i) for i in x.split(",")]),
    )
)

In [7]:
records = []
with foldcomp.open(selected_foldcomp) as db:
    for name, pdb in tqdm(db):
        name = name.split(".")[0]
        positions = seq_id_to_positions[name]
        residues = parse_pdb_str(name, pdb, positions)
        plddts = plddt_stat(pdb)
        if len(residues) < 3: continue
        
        sites = filter_by_clique(residues)
        if len(sites) != 0:
            records.append(
                {
                    "seq_id": name.split("-")[1],
                    "site": ";".join(sites),
                    "min_plddt": min(plddts)
                }
            )

100%|██████████| 610382/610382 [1:52:44<00:00, 90.23it/s]  


In [8]:
pd.DataFrame(records).to_csv(selected_pre_org_site, sep="\t", index=None)